In [6]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect("../database/n100.db")

In [7]:
tables = pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", conn)

tables

,name


In [8]:
"financial_ratios" in tables["name"].tolist()

False

In [9]:
import os

print(os.getcwd())

c:\Users\HP\OneDrive\Desktop\N100_Project\notebooks


In [10]:
import os

print(os.path.exists("../database/n100.db"))

True


In [11]:
import os

print(os.listdir("../database"))

['n100.db', 'nifty100.db']


In [12]:
conn = sqlite3.connect("../database/n100.db")

cursor = conn.cursor()

cursor.execute("""
SELECT count(*)
FROM sqlite_master
WHERE type='table'
""")

print(cursor.fetchone())

(0,)


In [13]:
import os

print(os.path.getsize("../database/n100.db"))

0


In [14]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect("../database/nifty100.db")

In [15]:
pl = pd.read_sql("SELECT * FROM profitandloss", conn)
bs = pd.read_sql("SELECT * FROM balancesheet", conn)
cf = pd.read_sql("SELECT * FROM cashflow", conn)

In [16]:
pl.columns = pl.iloc[0]
pl = pl.iloc[1:].reset_index(drop=True)

bs.columns = bs.iloc[0]
bs = bs.iloc[1:].reset_index(drop=True)

cf.columns = cf.iloc[0]
cf = cf.iloc[1:].reset_index(drop=True)

In [17]:
df = pl.merge(
    bs,
    on=["company_id", "year"],
    how="inner"
)

df = df.merge(
    cf,
    on=["company_id", "year"],
    how="inner"
)

df.head()

,id_x,company_id,year,sales,expenses,operating_profit,opm_percentage,other_income,interest,depreciation,...,fixed_assets,cwip,investments,other_asset,total_assets,id,operating_activity,investing_activity,financing_activity,net_cash_flow
0,61,ABB,Dec 2012,1653,1451,202,12,33,0,19,...,109,1,0,798,907,61,101,-59,-42,1
1,62,ABB,Mar 2014,2276,2009,267,12,49,0,22,...,98,1,0,1040,1139,62,155,-144,-42,-31
2,62,ABB,Mar 2014,2276,2009,267,12,49,0,22,...,98,1,0,1040,1139,73,0,0,0,0
3,63,ABB,Mar 2015,2289,1977,312,14,48,0,15,...,96,4,0,1274,1374,63,215,-187,-58,-30
4,63,ABB,Mar 2015,2289,1977,312,14,48,0,15,...,96,4,0,1274,1374,74,-35,-1864,1902,3


In [18]:
numeric_cols = [
    "sales",
    "operating_profit",
    "net_profit",
    "equity_share_capital",
    "reserves",
    "borrowings",
    "total_assets",
    "operating_activity",
    "interest",
    "dividend_payout"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

KeyError: 'equity_share_capital'

In [19]:
print(df.columns.tolist())

['id_x', 'company_id', 'year', 'sales', 'expenses', 'operating_profit', 'opm_percentage', 'other_income', 'interest', 'depreciation', 'profit_before_tax', 'tax_percentage', 'net_profit', 'eps', 'dividend_payout', 'id_y', 'equity_capital', 'reserves', 'borrowings', 'other_liabilities', 'total_liabilities', 'fixed_assets', 'cwip', 'investments', 'other_asset', 'total_assets', 'id', 'operating_activity', 'investing_activity', 'financing_activity', 'net_cash_flow']


In [20]:
for col in df.columns:
    print(col)

id_x
company_id
year
sales
expenses
operating_profit
opm_percentage
other_income
interest
depreciation
profit_before_tax
tax_percentage
net_profit
eps
dividend_payout
id_y
equity_capital
reserves
borrowings
other_liabilities
total_liabilities
fixed_assets
cwip
investments
other_asset
total_assets
id
operating_activity
investing_activity
financing_activity
net_cash_flow


In [21]:
df["return_on_equity_pct"] = np.where(
    (df["equity_capital"] + df["reserves"]) == 0,
    np.nan,
    (df["net_profit"] / (df["equity_capital"] + df["reserves"])) * 100
)

df[[
    "company_id",
    "year",
    "net_profit",
    "equity_capital",
    "reserves",
    "return_on_equity_pct"
]].head()

TypeError: operation 'rtruediv' not supported for dtype 'str' with dtype 'int64'

In [22]:
print(df[["equity_capital","reserves","borrowings","net_profit"]].dtypes)

0
equity_capital      str
reserves            str
borrowings          str
net_profit        int64
dtype: object


In [23]:
cols = [
    "equity_capital",
    "reserves",
    "borrowings",
    "net_profit"
]

for c in cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

In [24]:
print(df[cols].dtypes)

0
equity_capital    float64
reserves            int64
borrowings          int64
net_profit          int64
dtype: object


In [25]:
df["return_on_equity_pct"] = np.where(
    (df["equity_capital"] + df["reserves"]) == 0,
    np.nan,
    (df["net_profit"] / (df["equity_capital"] + df["reserves"])) * 100
)

df[[
    "company_id",
    "year",
    "return_on_equity_pct"
]].head()

,company_id,year,return_on_equity_pct
0,ABB,Dec 2012,22.411128
1,ABB,Mar 2014,25.126904
2,ABB,Mar 2014,25.126904
3,ABB,Mar 2015,24.439701
4,ABB,Mar 2015,24.439701


In [26]:
cols = [
    "borrowings",
    "equity_capital",
    "reserves",
    "total_liabilities"
]

for c in cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

In [27]:
df["debt_to_equity"] = np.where(
    (df["equity_capital"] + df["reserves"]) == 0,
    np.nan,
    df["borrowings"] / (df["equity_capital"] + df["reserves"])
)

df[[
    "company_id",
    "year",
    "borrowings",
    "equity_capital",
    "reserves",
    "debt_to_equity"
]].head()

,company_id,year,borrowings,equity_capital,reserves,debt_to_equity
0,ABB,Dec 2012,0,21.0,626,0.0
1,ABB,Mar 2014,0,21.0,767,0.0
2,ABB,Mar 2014,0,21.0,767,0.0
3,ABB,Mar 2015,0,21.0,916,0.0
4,ABB,Mar 2015,0,21.0,916,0.0


In [28]:
df["interest_coverage"] = np.where(
    df["interest"] == 0,
    np.nan,
    df["operating_profit"] / df["interest"]
)

df[[
    "company_id",
    "year",
    "operating_profit",
    "interest",
    "interest_coverage"
]].head()

TypeError: operation 'rtruediv' not supported for dtype 'str' with dtype 'float64'

In [29]:
print(df[["operating_profit","interest"]].dtypes)

0
operating_profit    float64
interest                str
dtype: object


In [30]:
df["interest"] = pd.to_numeric(df["interest"], errors="coerce")

print(df["interest"].dtype)

int64


In [31]:
df["interest_coverage"] = np.where(
    df["interest"] == 0,
    np.nan,
    df["operating_profit"] / df["interest"]
)

df[[
    "company_id",
    "year",
    "operating_profit",
    "interest",
    "interest_coverage"
]].head()

,company_id,year,operating_profit,interest,interest_coverage
0,ABB,Dec 2012,202.0,0,NaN
1,ABB,Mar 2014,267.0,0,NaN
2,ABB,Mar 2014,267.0,0,NaN
3,ABB,Mar 2015,312.0,0,NaN
4,ABB,Mar 2015,312.0,0,NaN


In [32]:
print(df["total_assets"].dtype)

str


In [33]:
df["total_assets"] = pd.to_numeric(df["total_assets"], errors="coerce")

In [34]:
df["asset_turnover"] = np.where(
    df["total_assets"] == 0,
    np.nan,
    df["sales"] / df["total_assets"]
)

df[[
    "company_id",
    "year",
    "sales",
    "total_assets",
    "asset_turnover"
]].head()

,company_id,year,sales,total_assets,asset_turnover
0,ABB,Dec 2012,1653,907,1.822492
1,ABB,Mar 2014,2276,1139,1.998244
2,ABB,Mar 2014,2276,1139,1.998244
3,ABB,Mar 2015,2289,1374,1.665939
4,ABB,Mar 2015,2289,1374,1.665939


In [35]:
df["free_cash_flow"] = (
    df["operating_activity"] +
    df["investing_activity"]
)

df[[
    "company_id",
    "year",
    "operating_activity",
    "investing_activity",
    "free_cash_flow"
]].head()

,company_id,year,operating_activity,investing_activity,free_cash_flow
0,ABB,Dec 2012,101,-59,101-59
1,ABB,Mar 2014,155,-144,155-144
2,ABB,Mar 2014,0,0,00
3,ABB,Mar 2015,215,-187,215-187
4,ABB,Mar 2015,-35,-1864,-35-1864


In [36]:
df["investing_activity"] = pd.to_numeric(df["investing_activity"], errors="coerce")

In [37]:
df["capex_ratio"] = np.where(
    df["sales"] == 0,
    np.nan,
    abs(df["investing_activity"]) / df["sales"]
)

df[[
    "company_id",
    "year",
    "sales",
    "investing_activity",
    "capex_ratio"
]].head()

,company_id,year,sales,investing_activity,capex_ratio
0,ABB,Dec 2012,1653,-59.0,0.035693
1,ABB,Mar 2014,2276,-144.0,0.063269
2,ABB,Mar 2014,2276,0.0,0.000000
3,ABB,Mar 2015,2289,-187.0,0.081695
4,ABB,Mar 2015,2289,-1864.0,0.814329


In [38]:
df["earnings_per_share"] = df["eps"]

df[[
    "company_id",
    "year",
    "eps",
    "earnings_per_share"
]].head()

,company_id,year,eps,earnings_per_share
0,ABB,Dec 2012,68,68
1,ABB,Mar 2014,93,93
2,ABB,Mar 2014,93,93
3,ABB,Mar 2015,108,108
4,ABB,Mar 2015,108,108


In [39]:
df["dividend_payout_ratio_pct"] = np.where(
    df["net_profit"] == 0,
    np.nan,
    (df["dividend_payout"] / df["net_profit"]) * 100
)

df[[
    "company_id",
    "year",
    "dividend_payout",
    "net_profit",
    "dividend_payout_ratio_pct"
]].head()

TypeError: operation 'truediv' not supported for dtype 'str' with dtype 'int64'

In [40]:
print(df[["dividend_payout","net_profit"]].dtypes)

0
dividend_payout      str
net_profit         int64
dtype: object


In [41]:
df["dividend_payout"] = pd.to_numeric(
    df["dividend_payout"],
    errors="coerce"
)

print(df["dividend_payout"].dtype)

float64


In [42]:
df["net_profit"] = pd.to_numeric(
    df["net_profit"],
    errors="coerce"
)

print(df["net_profit"].dtype)

int64


In [43]:
cols = ["dividend_payout", "net_profit"]

for c in cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print(df[cols].dtypes)

0
dividend_payout    float64
net_profit           int64
dtype: object


In [44]:
ratio_cols = [
    "company_id",
    "year",
    "return_on_equity_pct",
    "debt_to_equity",
    "interest_coverage",
    "asset_turnover",
    "dividend_payout_ratio"
]

financial_ratios = df[ratio_cols]

financial_ratios.head()

KeyError: "['dividend_payout_ratio'] not in index"

In [45]:
for col in df.columns:
    if "dividend" in col.lower():
        print(col)

dividend_payout


In [46]:
ratio_cols = [
    "company_id",
    "year",
    "return_on_equity_pct",
    "debt_to_equity",
    "interest_coverage",
    "asset_turnover",
    "dividend_payout"
]

In [47]:
df["dividend_payout_ratio"] = np.where(
    df["net_profit"] == 0,
    np.nan,
    (df["dividend_payout"] / df["net_profit"]) * 100
)

df[[
    "company_id",
    "year",
    "dividend_payout",
    "net_profit",
    "dividend_payout_ratio"
]].head()

,company_id,year,dividend_payout,net_profit,dividend_payout_ratio
0,ABB,Dec 2012,25.0,145,17.241379
1,ABB,Mar 2014,25.0,198,12.626263
2,ABB,Mar 2014,25.0,198,12.626263
3,ABB,Mar 2015,29.0,229,12.663755
4,ABB,Mar 2015,29.0,229,12.663755


In [48]:
for col in df.columns:
    if "dividend" in col.lower():
        print(col)

dividend_payout
dividend_payout_ratio


In [49]:
ratio_cols = [
    "company_id",
    "year",
    "return_on_equity_pct",
    "debt_to_equity",
    "interest_coverage",
    "asset_turnover",
    "dividend_payout_ratio"
]

financial_ratios = df[ratio_cols]

financial_ratios.head()

,company_id,year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,dividend_payout_ratio
0,ABB,Dec 2012,22.411128,0.0,NaN,1.822492,17.241379
1,ABB,Mar 2014,25.126904,0.0,NaN,1.998244,12.626263
2,ABB,Mar 2014,25.126904,0.0,NaN,1.998244,12.626263
3,ABB,Mar 2015,24.439701,0.0,NaN,1.665939,12.663755
4,ABB,Mar 2015,24.439701,0.0,NaN,1.665939,12.663755


In [50]:
financial_ratios.to_csv(
    "../reports/financial_ratios_day12.csv",
    index=False
)

print("Financial Ratios Report Saved!")

Financial Ratios Report Saved!


In [51]:
import os

print(os.listdir("../reports"))

['benchmark_companies.csv', 'benchmark_companies_day7.csv', 'capital_allocation_day11.csv', 'financial_ratios_day12.csv', 'highest_price_stocks.csv', 'peer_group_distribution.csv', 'sector_distribution.csv', 'top_market_cap_companies.csv', 'top_market_cap_day7.csv', 'top_roce_companies.csv', 'top_roe_companies.csv', 'top_volume_stocks.csv']
